In [3]:
import requests
import pandas as pd

url = "https://api.ratings.food.gov.uk/Establishments"
headers = {
    "x-api-version": "2",
    "accept": "application/json"
}
params = {
    "longitude": -0.1340,   # Soho大致中心
    "latitude": 51.5138,
    "maxDistanceLimit": 1,  # 单位:英里,先拉1英里范围
    "pageSize": 5000
}
r = requests.get(url, headers=headers, params=params)
print(r.status_code)
data = r.json()
print(data.keys())          # 看看实际有哪些key

200
dict_keys(['establishments', 'meta', 'links'])


In [4]:
data = r.json()
df = pd.json_normalize(data["establishments"])
df.to_csv("westminster_fhrs_raw.csv", index=False)  # 存一份原始的

print(df.shape)                          # 一共多少条
print(df["BusinessType"].value_counts()) # 各类型分布
print(df.columns.tolist())               # 看字段名,确认经纬度和地址字段叫什么

(4538, 28)
BusinessType
Restaurant/Cafe/Canteen                  2858
Retailers - other                         728
Takeaway/sandwich shop                    292
Pub/bar/nightclub                         195
Hotel/bed & breakfast/guest house         154
Other catering premises                   121
Retailers - supermarkets/hypermarkets      45
School/college/university                  40
Importers/Exporters                        37
Caring Premises                            24
Manufacturers/packers                      19
Mobile caterer                             15
Distributors/Transporters                  10
Name: count, dtype: int64
['AddressLine1', 'AddressLine2', 'AddressLine3', 'AddressLine4', 'BusinessName', 'BusinessType', 'BusinessTypeID', 'ChangesByServerID', 'Distance', 'FHRSID', 'LocalAuthorityBusinessID', 'LocalAuthorityCode', 'LocalAuthorityEmailAddress', 'LocalAuthorityName', 'LocalAuthorityWebSite', 'NewRatingPending', 'Phone', 'PostCode', 'RatingDate', 'RatingKey',

In [6]:
df["lon"] = df["geocode.longitude"].astype(float)
df["lat"] = df["geocode.latitude"].astype(float)

# 用你自己road network的bbox替换下面四个值
# 如果不记得,在你之前OSMnx下载的图上跑:
#   nodes = ox.graph_to_gdfs(G, edges=False)
#   minx, miny, maxx, maxy = nodes.total_bounds
df_soho = df[(df.lon.between(-0.1415564, -0.1295152)) & (df.lat.between(51.5102453, 51.5162063))]

food_types = ["Restaurant/Cafe/Canteen", "Takeaway/sandwich shop", "Other catering premises"]
df_food = df_soho[df_soho["BusinessType"].isin(food_types)]

print("Soho范围内总记录:", len(df_soho))
print("Soho范围内餐饮POI:", len(df_food))
print(df_food["BusinessType"].value_counts())

Soho范围内总记录: 866
Soho范围内餐饮POI: 704
BusinessType
Restaurant/Cafe/Canteen    652
Takeaway/sandwich shop      40
Other catering premises     12
Name: count, dtype: int64


In [7]:
addr = (df_food["AddressLine1"].fillna("") + " " + df_food["AddressLine2"].fillna(""))
suspects = df_food[addr.str.contains(
    "Unit|Basement|Rear|Kitchen|Studio|Arch|Floor", case=False, na=False
)]
print("嫌疑dark kitchen数:", len(suspects))
print(suspects[["BusinessName", "AddressLine1", "AddressLine2", "BusinessType"]].to_string())


嫌疑dark kitchen数: 202
                                   BusinessName                                                                 AddressLine1           AddressLine2             BusinessType
64                            ALL'ANTICO VINAIO                              BASEMENT AND GROUND FLOOR 61 OLD COMPTON STREET                          Takeaway/sandwich shop
65                              ALTA Restaurant                                            UNIT G9 KINGLY COURT KINGLY COURT                         Restaurant/Cafe/Canteen
84                               Andrew Edmunds                                BASEMENT AND GROUND FLOOR 46 LEXINGTON STREET                         Restaurant/Cafe/Canteen
105                               Archer Street                                 BASEMENT AND GROUND FLOORS 3-4 ARCHER STREET                         Restaurant/Cafe/Canteen
143                          B Bagel Bakery Bar                                         GROUND FLOOR SOUTH 54 WARD

In [8]:
addr = (df_food["AddressLine1"].fillna("") + " " + df_food["AddressLine2"].fillna(""))

# 去掉 Basement/Floor 这些无区分度的词,只留真正指向"非临街独立单元"的信号
suspects = df_food[addr.str.contains(
    r"\bUnit\b|\bArch\b|\bRear\b|Industrial|Estate|Warehouse", 
    case=False, na=False, regex=True
)]
print("收紧后嫌疑数:", len(suspects))
print(suspects[["BusinessName", "AddressLine1", "AddressLine2", "BusinessType"]].to_string())

收紧后嫌疑数: 12
                        BusinessName                                             AddressLine1 AddressLine2             BusinessType
65                   ALTA Restaurant                        UNIT G9 KINGLY COURT KINGLY COURT               Restaurant/Cafe/Canteen
157                    Bagel Factory           UNIT 1 OXFORD CIRCUS STATION 237 OXFORD STREET                Takeaway/sandwich shop
725                 Donia Restaurant                      UNIT 2.1 KINGLY COURT KINGLY STREET               Restaurant/Cafe/Canteen
1111            Imads Syrian Kitchen           SECOND FLOOR UNIT 14 KINGLY COURT KINGLY COURT               Restaurant/Cafe/Canteen
1207                          Kapara                   UNIT 2 ILONA ROSE HOUSE MANETTE STREET               Restaurant/Cafe/Canteen
1254                 Kova Patisserie                              UNIT 5 9-12 ST ANNE'S COURT               Restaurant/Cafe/Canteen
1707  Paradiso Burger & Cocktail Bar           GROUND FLOOR UNIT 

In [9]:
other = df_food[df_food["BusinessType"] == "Other catering premises"]
print("Other catering premises数:", len(other))
print(other[["BusinessName", "AddressLine1", "AddressLine2"]].to_string())

Other catering premises数: 12
                                                          BusinessName                           AddressLine1 AddressLine2
8                                                          20 Air cafe                          20 AIR STREET             
856                                                          Fooditude       SECOND FLOOR 30 BROADWICK STREET             
858                                                          Fooditude         KNIGHTWAY HOUSE 20 SOHO SQUARE             
1426                                                           Mamapen               21 GREAT PULTENEY STREET             
1949                                                Rocket at Duolingo                     141 WARDOUR STREET             
2501                                                  The Soho Kitchen                           1 SOHO PLACE  WESTMINSTER
2610                                                          Vacherin       GROUND FLOOR SOUTH 23 SAVILE ROW 

In [10]:
# 剔除明显的contract catering(不产生外卖courier的企业内部供餐)
exclude = df_food["BusinessName"].str.contains(
    "Vacherin|Fooditude|Duolingo|Pinterest|Burberry|Marshall Street", 
    case=False, na=False
)
df_food_clean = df_food[~exclude].copy()

# 存成最终POI,带上模型需要的字段
cols = ["BusinessName", "BusinessType", "AddressLine1", "PostCode",
        "lon", "lat", "RatingValue", "FHRSID"]
df_food_clean[cols].to_csv("soho_food_pois.csv", index=False)
print("最终餐饮POI数:", len(df_food_clean))

最终餐饮POI数: 697


Opening hours 

In [12]:
import osmnx as ox

tags = {"amenity": ["restaurant", "cafe", "fast_food", "pub", "bar"]}
gdf = ox.features_from_bbox(bbox=(-0.1415564, 51.5102453, -0.1295152, 51.5162063), tags=tags)

# 关键字段
cols = ["name", "amenity", "cuisine", "opening_hours", "takeaway", "delivery", "geometry"]
gdf_oh = gdf[[c for c in cols if c in gdf.columns]]
print("有opening_hours的比例:", gdf_oh["opening_hours"].notna().mean())

有opening_hours的比例: 0.31901840490797545


In [13]:
for c in ["takeaway", "delivery", "cuisine", "name"]:
    if c in gdf_oh.columns:
        print(f"{c} 覆盖率:", round(gdf_oh[c].notna().mean(), 3))
print("OSM餐饮POI总数:", len(gdf_oh))

takeaway 覆盖率: 0.158
delivery 覆盖率: 0.012
cuisine 覆盖率: 0.6
name 覆盖率: 0.994
OSM餐饮POI总数: 652


In [14]:
has_oh = gdf_oh[gdf_oh["opening_hours"].notna()]
no_oh = gdf_oh[gdf_oh["opening_hours"].isna()]
print("有营业时间的类型分布:\n", has_oh["amenity"].value_counts())
print("\n无营业时间的类型分布:\n", no_oh["amenity"].value_counts())

有营业时间的类型分布:
 amenity
restaurant    114
cafe           30
bar            24
fast_food      23
pub            17
Name: count, dtype: int64

无营业时间的类型分布:
 amenity
restaurant    253
cafe           59
fast_food      52
bar            44
pub            36
Name: count, dtype: int64
